---
**This notebook is `%run` automatically by the `(MAIN)` notebook when `VERSION = 'HARDWARE'`.**
It reads `SCOPETYPE`, `PLATFORM`, `SS_VER` from the MAIN notebook, connects to the ChipWhisperer, builds and flashes the `basic-passwdcheck` firmware, and defines `cap_pass_trace()`.

---

Defaults from MAIN are `SCOPETYPE='OPENADC'`, `PLATFORM='CWHUSKY'`, `SS_VER='SS_VER_2_1'` - correct for the CW-Husky kit (SAM4S target). For other rigs change them in MAIN: `CWLITEARM`/`CW308_STM32F3` for an STM32F3, `CWLITEXMEGA` for XMEGA, `CWNANO` for the CW-Nano.

In [ ]:
# SCOPETYPE / PLATFORM / SS_VER come from the MAIN notebook that %run this file.
print('setup:', SCOPETYPE, PLATFORM, SS_VER)

Connect the scope and target and pick the right programmer (`Setup_Generic` handles the SAM4S/Husky case):

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

The following code will build the firmware for the target.

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/basic-passwdcheck
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

Finally, all that's left is to program the device, which can be done with the following line:

In [ ]:
cw.program_target(scope, prog, "../../../firmware/mcu/basic-passwdcheck/basic-passwdcheck-{}.hex".format(PLATFORM))

To make interacting with the hardware easier, let's define a function to attempt a password and return a power trace:

In [ ]:
def cap_pass_trace(pass_guess):
    reset_target(scope)
    num_char = target.in_waiting()
    while num_char > 0:
        target.read(num_char, 10)
        time.sleep(0.01)
        num_char = target.in_waiting()

    scope.arm()
    target.write(pass_guess)
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')

    trace = scope.get_last_trace()
    return trace

We also don't need all of the default 5000 samples in the trace. 3000 is a good starting point for most targets:

In [ ]:
scope.adc.samples = 3000